In [1]:
# ── Cell 1: Clone repo ──
import os
os.chdir('/kaggle/working')
!git clone https://github.com/srikant31/Gravitational-Wave-Classification-CNN.git
os.chdir('/kaggle/working/Gravitational-Wave-Classification-CNN')
!git checkout version-2
!git pull

# ── Cell 2: Install deps ──
!pip install -q scikit-image scikit-learn image-classifiers

# ── Cell 3: Confirm GPU ──
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

# ── Cell 4: Build the dataset (this is what actually creates the train/val/test folders) ──
import pathlib
from pywavecnn.data import prepare_dataset
from pywavecnn.config import TrainingConfig, TRAIN_DIR, VALID_DIR, TEST_DIR

prepare_dataset(TrainingConfig())

train_dir = pathlib.Path(TRAIN_DIR)
valid_dir = pathlib.Path(VALID_DIR)
test_dir  = pathlib.Path(TEST_DIR)

print("TRAIN_DIR:", TRAIN_DIR)
print("Exists?", train_dir.exists())

# ── Cell 5: EDA — real counts + statistical reliability check ──
import numpy as np
import pandas as pd
from scipy import stats

class_names = sorted([p.name for p in train_dir.iterdir() if p.is_dir()])

counts = pd.DataFrame({
    'train': [len(list((train_dir / c).glob("*.png"))) for c in class_names],
    'val':   [len(list((valid_dir / c).glob("*.png"))) for c in class_names],
    'test':  [len(list((test_dir  / c).glob("*.png"))) for c in class_names],
}, index=class_names)
counts['total'] = counts.sum(axis=1)
counts = counts.sort_values('test')

print("\n=== Actual per-class counts ===")
print(counts.to_string())

def wilson_ci(successes, n, confidence=0.95):
    if n == 0:
        return (np.nan, np.nan)
    return stats.binomtest(successes, n).proportion_ci(confidence_level=confidence, method='wilson')

results = []
for cls in counts.index:
    n_test = counts.loc[cls, 'test']
    if n_test == 0:
        results.append({'class': cls, 'test_n': 0, 'recall_granularity_%': np.nan, 'ci_width_at_50pct_recall_%': np.nan})
        continue
    granularity = 100 / n_test
    successes_at_50pct = round(n_test * 0.5)
    ci_low, ci_high = wilson_ci(successes_at_50pct, n_test)
    ci_width = (ci_high - ci_low) * 100
    results.append({
        'class': cls, 'test_n': n_test,
        'recall_granularity_%': round(granularity, 1),
        'ci_width_at_50pct_recall_%': round(ci_width, 1)
    })

eda_df = pd.DataFrame(results)
print("\n=== Recall reliability by class (sorted smallest test N first) ===")
print(eda_df.to_string(index=False))

UNRELIABLE_THRESHOLD_N = 30
unreliable = eda_df[eda_df['test_n'] < UNRELIABLE_THRESHOLD_N]
print(f"\n=== Classes with test_n < {UNRELIABLE_THRESHOLD_N} (recall here is NOT a reliable target metric) ===")
print(unreliable.to_string(index=False))

print(f"\nTotal classes: {len(class_names)}")
print(f"Classes below reliability threshold: {len(unreliable)} ({len(unreliable)/len(class_names)*100:.0f}% of all classes)")

Cloning into 'Gravitational-Wave-Classification-CNN'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 45 (delta 4), reused 44 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (45/45), 6.63 MiB | 21.57 MiB/s, done.
Resolving deltas: 100% (4/4), done.
Branch 'version-2' set up to track remote branch 'version-2' from 'origin'.
Switched to a new branch 'version-2'
Already up to date.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 2.1 MB/s eta 0:00:00
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
trainingsetv1d1.tar.gz: 5.12 GB downloaded...
trainingset_v1d1_metadata.csv: 0.01 GB downloaded...
TRAIN_DIR: /kaggle/working/Gravitational-Wave-Classification-CNN/data/train
Exists? True

=== Actual per-class counts ===
                     train  val  test  total
Paired_Doves            86   

In [ ]:
# ── Cell 1: Clone repo ──
import os
os.chdir('/kaggle/working')
!git clone https://github.com/srikant31/Gravitational-Wave-Classification-CNN.git
os.chdir('/kaggle/working/Gravitational-Wave-Classification-CNN')
!git checkout version-2
!git pull

# ── Cell 2: Install deps ──
!pip install -q scikit-image scikit-learn image-classifiers

# ── Cell 3: Confirm GPU ──
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

# ── Cell 4: Clean any stale cache from previous runs ──
import shutil, glob
for pattern in ["/kaggle/working/train_cache*", "/kaggle/working/valid_cache*"]:
    for f in glob.glob(pattern):
        try:
            if os.path.isdir(f):
                shutil.rmtree(f)
            else:
                os.remove(f)
            print(f"Removed: {f}")
        except Exception as e:
            print(f"Could not remove {f}: {e}")

# ── Cell 5: Main script ──
import time, pathlib, numpy as np, pandas as pd
from tensorflow.keras import layers, regularizers
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
from pywavecnn.data import prepare_dataset
from pywavecnn.config import TrainingConfig, TRAIN_DIR, VALID_DIR, TEST_DIR

tf.random.set_seed(220301)
IMG_SIZE, BATCH, EPOCHS, SEED = (128, 128), 32, 15, 220301
autotune = tf.data.AUTOTUNE

# ---- Build the dataset ONCE, reuse across all variants ----
prepare_dataset(TrainingConfig())
data_dir = pathlib.Path(TRAIN_DIR)
class_names = sorted([p.name for p in data_dir.iterdir() if p.is_dir()])
num_classes = len(class_names)

train_ds_raw = tf.keras.utils.image_dataset_from_directory(TRAIN_DIR, seed=SEED, image_size=IMG_SIZE, batch_size=BATCH)
valid_ds_raw = tf.keras.utils.image_dataset_from_directory(VALID_DIR, seed=SEED, image_size=IMG_SIZE, batch_size=BATCH)
test_ds = tf.keras.utils.image_dataset_from_directory(TEST_DIR, seed=SEED, image_size=IMG_SIZE, batch_size=BATCH)

train_ds = train_ds_raw.map(lambda x, y: (x, tf.one_hot(y, num_classes)), num_parallel_calls=autotune)
valid_ds = valid_ds_raw.map(lambda x, y: (x, tf.one_hot(y, num_classes)), num_parallel_calls=autotune)
train_ds = train_ds.cache("/kaggle/working/train_cache").prefetch(autotune)
valid_ds = valid_ds.cache("/kaggle/working/valid_cache").prefetch(autotune)
test_ds = test_ds.prefetch(autotune)

# ---- NEW: Real per-class test counts, so recall swings are interpretable ----
test_dir_path = pathlib.Path(TEST_DIR)
test_counts = {name: len(list((test_dir_path / name).glob("*.png"))) for name in class_names}
print("\n=== Test set image counts per class (sorted smallest first) ===")
for name, count in sorted(test_counts.items(), key=lambda x: x[1]):
    print(f"  {name}: {count}  (1 correct prediction = {100/count:.1f} recall points)")

# ---- Class weights, capped, as before ----
train_labels_list = []
for i, name in enumerate(class_names):
    train_labels_list.extend([i] * len(list((data_dir / name).glob("*.png"))))
train_labels_arr = np.array(train_labels_list)
raw_weights = compute_class_weight(class_weight="balanced", classes=np.unique(train_labels_arr), y=train_labels_arr)

try:
    from classification_models.tfkeras import Classifiers
    ResNetBackbone, preprocess_input = Classifiers.get('resnet18')
    BACKBONE_NAME = "resnet18"
except Exception:
    from tensorflow.keras.applications import ResNet50 as ResNetBackbone
    from tensorflow.keras.applications.resnet50 import preprocess_input
    BACKBONE_NAME = "resnet50_fallback"

# ---- NEW: Focal loss implementation ----
class FocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, label_smoothing=0.1, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def call(self, y_true, y_pred):
        y_true = y_true * (1 - self.label_smoothing) + self.label_smoothing / tf.cast(tf.shape(y_true)[-1], tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        ce = -y_true * tf.math.log(y_pred)
        weight = tf.pow(1 - y_pred, self.gamma)
        return tf.reduce_sum(weight * ce, axis=-1)

# ---- UPDATED: configs, including 2 new variants for recall improvement ----
CONFIGS = {
    "control_class_weighted_ls":  dict(dropout=0.0, l2=0.0,    clipnorm=None, lr_schedule=False, weight_cap=10.0, use_focal=False),
    "plus_dropout":                dict(dropout=0.3, l2=0.0,    clipnorm=None, lr_schedule=False, weight_cap=10.0, use_focal=False),
    "plus_l2":                     dict(dropout=0.0, l2=1e-4,   clipnorm=None, lr_schedule=False, weight_cap=10.0, use_focal=False),
    "plus_grad_clip":              dict(dropout=0.0, l2=0.0,    clipnorm=1.0,  lr_schedule=False, weight_cap=10.0, use_focal=False),
    "plus_lr_schedule":            dict(dropout=0.0, l2=0.0,    clipnorm=None, lr_schedule=True,  weight_cap=10.0, use_focal=False),
    "plus_all_combined":           dict(dropout=0.3, l2=1e-4,   clipnorm=1.0,  lr_schedule=True,  weight_cap=10.0, use_focal=False),
    # NEW: same base as your best performer, but with a higher weight cap
    "plus_grad_clip_weight25":     dict(dropout=0.0, l2=0.0,    clipnorm=1.0,  lr_schedule=False, weight_cap=25.0, use_focal=False),
    # NEW: same base, but using focal loss instead of weighted cross-entropy
    "plus_grad_clip_focal":        dict(dropout=0.0, l2=0.0,    clipnorm=1.0,  lr_schedule=False, weight_cap=10.0, use_focal=True),
}

def build_model(cfg):
    if BACKBONE_NAME == "resnet18":
        base = ResNetBackbone(input_shape=IMG_SIZE + (3,), weights='imagenet', include_top=False)
    else:
        base = ResNetBackbone(include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,))
    base.trainable = False
    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = preprocess_input(inputs)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    reg = regularizers.l2(cfg["l2"]) if cfg["l2"] > 0 else None
    x = layers.Dense(128, activation="relu", kernel_regularizer=reg)(x)
    if cfg["dropout"] > 0:
        x = layers.Dropout(cfg["dropout"])(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs)

results = []
for name, cfg in CONFIGS.items():
    print(f"\n{'='*60}\nRunning: {name}  ({cfg})\n{'='*60}")
    model = build_model(cfg)

    # weight cap is now per-config
    capped_weights = np.clip(raw_weights, None, raw_weights.min() * cfg["weight_cap"])
    class_weight_dict = dict(enumerate(capped_weights))

    optimizer_kwargs = {"learning_rate": 1e-3}
    if cfg["clipnorm"] is not None:
        optimizer_kwargs["clipnorm"] = cfg["clipnorm"]
    optimizer = tf.keras.optimizers.RMSprop(**optimizer_kwargs)

    loss_fn = FocalLoss(gamma=2.0, label_smoothing=0.3) if cfg["use_focal"] else \
              tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.3)

    model.compile(loss=loss_fn, optimizer=optimizer, metrics=["accuracy"])

    callbacks = [tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True)]
    if cfg["lr_schedule"]:
        callbacks.append(tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6))

    t0 = time.time()
    hist = model.fit(train_ds, validation_data=valid_ds, epochs=EPOCHS,
                      class_weight=class_weight_dict, callbacks=callbacks, verbose=2)
    train_time = time.time() - t0

    y_true, y_pred = [], []
    for imgs, labels in test_ds:
        preds = model.predict(imgs, verbose=0)
        y_true.extend(labels.numpy())
        y_pred.extend(np.argmax(preds, axis=1))
    report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)
    overfit_gap = hist.history["accuracy"][-1] - hist.history["val_accuracy"][-1]

    row = {
        "variant": name, "epochs_run": len(hist.history["accuracy"]), "train_time_sec": round(train_time, 1),
        "test_accuracy": round(report["accuracy"], 4), "macro_f1": round(report["macro avg"]["f1-score"], 4),
        "overfit_gap": round(overfit_gap, 4),
        "blip_recall": round(report.get("Blip", {}).get("recall", float("nan")), 2),
        "no_glitch_recall": round(report.get("No_Glitch", {}).get("recall", float("nan")), 2),
        "paired_doves_recall": round(report.get("Paired_Doves", {}).get("recall", float("nan")), 2),
        "paired_doves_precision": round(report.get("Paired_Doves", {}).get("precision", float("nan")), 2),
    }
    results.append(row)
    print(f"\n=== RESULT: {name} ===")
    print(row)

print("\n\n" + "="*80)
print("FINAL COMPARISON TABLE — all 8 variants")
print("="*80)
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
results_df.to_csv("/kaggle/working/regularization_comparison.csv", index=False)

In [ ]:
# ── Cell 1: Clone repo ──
import os
os.chdir('/kaggle/working')
!git clone https://github.com/srikant31/Gravitational-Wave-Classification-CNN.git
os.chdir('/kaggle/working/Gravitational-Wave-Classification-CNN')
!git checkout version-2
!git pull

# ── Cell 2: Install deps ──
!pip install -q scikit-image scikit-learn image-classifiers

# ── Cell 3: Clean stale cache ──
import shutil, glob
for pattern in ["/kaggle/working/train_cache*", "/kaggle/working/valid_cache*"]:
    for f in glob.glob(pattern):
        try:
            shutil.rmtree(f) if os.path.isdir(f) else os.remove(f)
        except Exception:
            pass

# ── Cell 4: Multi-seed script — tests whether your candidates are REAL improvements ──
import time, pathlib, numpy as np, pandas as pd, tensorflow as tf
from tensorflow.keras import layers, regularizers
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
from pywavecnn.data import prepare_dataset
from pywavecnn.config import TrainingConfig, TRAIN_DIR, VALID_DIR, TEST_DIR

IMG_SIZE, BATCH, EPOCHS = (128, 128), 32, 15
autotune = tf.data.AUTOTUNE
SEEDS = [220301, 42, 7]   # 3 seeds per config — enough to see if results are stable

prepare_dataset(TrainingConfig())
data_dir = pathlib.Path(TRAIN_DIR)
class_names = sorted([p.name for p in data_dir.iterdir() if p.is_dir()])
num_classes = len(class_names)

train_labels_list = []
for i, name in enumerate(class_names):
    train_labels_list.extend([i] * len(list((data_dir / name).glob("*.png"))))
train_labels_arr = np.array(train_labels_list)
raw_weights = compute_class_weight(class_weight="balanced", classes=np.unique(train_labels_arr), y=train_labels_arr)

try:
    from classification_models.tfkeras import Classifiers
    ResNetBackbone, preprocess_input = Classifiers.get('resnet18')
    BACKBONE_NAME = "resnet18"
except Exception:
    from tensorflow.keras.applications import ResNet50 as ResNetBackbone
    from tensorflow.keras.applications.resnet50 import preprocess_input
    BACKBONE_NAME = "resnet50_fallback"

# ── Only your top candidates — the winner, plus the two ideas you want verified ──
CONFIGS = {
    "plus_lr_schedule":       dict(dropout=0.0, l2=0.0, clipnorm=None, lr_schedule=True,  weight_cap=10.0, use_focal=False),
    "plus_grad_clip_weight25": dict(dropout=0.0, l2=0.0, clipnorm=1.0, lr_schedule=False, weight_cap=25.0, use_focal=False),
    "plus_grad_clip_focal":    dict(dropout=0.0, l2=0.0, clipnorm=1.0, lr_schedule=False, weight_cap=10.0, use_focal=True),
}

class FocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, label_smoothing=0.1, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.label_smoothing = label_smoothing
    def call(self, y_true, y_pred):
        y_true = y_true * (1 - self.label_smoothing) + self.label_smoothing / tf.cast(tf.shape(y_true)[-1], tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        ce = -y_true * tf.math.log(y_pred)
        weight = tf.pow(1 - y_pred, self.gamma)
        return tf.reduce_sum(weight * ce, axis=-1)

def build_model(cfg):
    if BACKBONE_NAME == "resnet18":
        base = ResNetBackbone(input_shape=IMG_SIZE + (3,), weights='imagenet', include_top=False)
    else:
        base = ResNetBackbone(include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,))
    base.trainable = False
    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = preprocess_input(inputs)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    reg = regularizers.l2(cfg["l2"]) if cfg["l2"] > 0 else None
    x = layers.Dense(128, activation="relu", kernel_regularizer=reg)(x)
    if cfg["dropout"] > 0:
        x = layers.Dropout(cfg["dropout"])(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs)

all_results = []
for name, cfg in CONFIGS.items():
    for seed in SEEDS:
        print(f"\n{'='*60}\n{name} — seed {seed}\n{'='*60}")
        tf.random.set_seed(seed)

        train_ds_raw = tf.keras.utils.image_dataset_from_directory(TRAIN_DIR, seed=seed, image_size=IMG_SIZE, batch_size=BATCH)
        valid_ds_raw = tf.keras.utils.image_dataset_from_directory(VALID_DIR, seed=seed, image_size=IMG_SIZE, batch_size=BATCH)
        test_ds = tf.keras.utils.image_dataset_from_directory(TEST_DIR, seed=seed, image_size=IMG_SIZE, batch_size=BATCH)

        train_ds = train_ds_raw.map(lambda x, y: (x, tf.one_hot(y, num_classes)), num_parallel_calls=autotune).prefetch(autotune)
        valid_ds = valid_ds_raw.map(lambda x, y: (x, tf.one_hot(y, num_classes)), num_parallel_calls=autotune).prefetch(autotune)
        test_ds = test_ds.prefetch(autotune)

        capped_weights = np.clip(raw_weights, None, raw_weights.min() * cfg["weight_cap"])
        class_weight_dict = dict(enumerate(capped_weights))

        model = build_model(cfg)
        optimizer_kwargs = {"learning_rate": 1e-3}
        if cfg["clipnorm"] is not None:
            optimizer_kwargs["clipnorm"] = cfg["clipnorm"]
        optimizer = tf.keras.optimizers.RMSprop(**optimizer_kwargs)
        loss_fn = FocalLoss(gamma=2.0, label_smoothing=0.3) if cfg["use_focal"] else \
                  tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.3)
        model.compile(loss=loss_fn, optimizer=optimizer, metrics=["accuracy"])

        callbacks = [tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True)]
        if cfg["lr_schedule"]:
            callbacks.append(tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6))

        hist = model.fit(train_ds, validation_data=valid_ds, epochs=EPOCHS,
                          class_weight=class_weight_dict, callbacks=callbacks, verbose=0)

        y_true, y_pred = [], []
        for imgs, labels in test_ds:
            preds = model.predict(imgs, verbose=0)
            y_true.extend(labels.numpy())
            y_pred.extend(np.argmax(preds, axis=1))
        report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)

        all_results.append({
            "variant": name, "seed": seed,
            "test_accuracy": round(report["accuracy"], 4),
            "macro_f1": round(report["macro avg"]["f1-score"], 4),
            "paired_doves_recall": round(report.get("Paired_Doves", {}).get("recall", float("nan")), 2),
            "no_glitch_recall": round(report.get("No_Glitch", {}).get("recall", float("nan")), 2),
        })
        print(all_results[-1])

# ── Aggregate: mean ± std per config — THIS tells you what's real vs noise ──
results_df = pd.DataFrame(all_results)
print("\n\n" + "="*80)
print("MEAN ± STD ACROSS SEEDS — a config only 'wins' if its mean beats others by more than 1 std")
print("="*80)
summary = results_df.groupby("variant").agg(['mean', 'std'])
print(summary.to_string())
results_df.to_csv("/kaggle/working/multiseed_comparison.csv", index=False)

In [ ]:
import pathlib, os

data_dir = pathlib.Path(TRAIN_DIR)
class_names = sorted([p.name for p in data_dir.iterdir() if p.is_dir()])

sample_class = class_names[0]
sample_path = data_dir / sample_class
entries = list(sample_path.iterdir())[:10]

print(f"=== Structure check under: {sample_path} ===")
for e in entries:
    print(f"  {e.name}  [{'DIR' if e.is_dir() else 'FILE'}]")

DURATION_TAGS = {'0.5', '1.0', '2.0', '4.0'}

def analyze_class(class_dir):
    all_files = []
    for root, dirs, files in os.walk(class_dir):
        for f in files:
            if f.lower().endswith('.png'):
                all_files.append(os.path.join(root, f))
    total_files = len(all_files)
    unique_event_ids = set()
    for fpath in all_files:
        stem = pathlib.Path(fpath).stem
        parent_name = pathlib.Path(fpath).parent.name
        if stem in DURATION_TAGS:
            unique_event_ids.add(parent_name)
        else:
            matched = False
            for tag in DURATION_TAGS:
                if stem.endswith(tag):
                    unique_event_ids.add(stem[:-len(tag)].rstrip('_'))
                    matched = True
                    break
            if not matched:
                unique_event_ids.add(stem)
    return total_files, len(unique_event_ids)

print("\n=== Per-class: total image files vs estimated unique events ===")
for cls in class_names:
    total, unique_events = analyze_class(data_dir / cls)
    ratio = round(total / unique_events, 2) if unique_events > 0 else float('nan')
    print(f"  {cls}: {total} image files, ~{unique_events} unique events, ratio={ratio}")

print("\n=== Interpretation ===")
print("ratio ≈ 4.0 → 4 duration views per event are being used as separate images")
print("ratio ≈ 1.0 → only 1 view per event is being used")

In [ ]:
import pathlib
import pandas as pd

tar_dirs = {'train': TRAIN_DIR, 'val': VALID_DIR, 'test': TEST_DIR}
class_names = sorted([p.name for p in pathlib.Path(TRAIN_DIR).iterdir() if p.is_dir()])

tar_counts = {}
for cls in class_names:
    total = 0
    for split_dir in tar_dirs.values():
        cls_path = pathlib.Path(split_dir) / cls
        if cls_path.exists():
            total += len(list(cls_path.rglob("*.png")))
    tar_counts[cls] = total

tar_df = pd.Series(tar_counts, name='total_image_files').sort_values()
print("=== Original .tar dataset — per-class totals (train+val+test) ===")
print(tar_df.to_string())

# ── Set this from Cell 1's output ──
VIEW_MULTIPLIER = 1   # change to 4 if Cell 1 showed ratio ≈ 4.0

tar_df_corrected = (tar_df / VIEW_MULTIPLIER).round(0).astype(int)
tar_df_corrected.name = 'estimated_unique_events'

print(f"\n=== Corrected to unique events (÷{VIEW_MULTIPLIER}) ===")
print(tar_df_corrected.to_string())

print(f"\nSmallest class: {tar_df_corrected.idxmin()} ({tar_df_corrected.min()} unique events)")
print(f"Largest class: {tar_df_corrected.idxmax()} ({tar_df_corrected.max()} unique events)")
print(f"Imbalance ratio: {tar_df_corrected.max()/tar_df_corrected.min():.1f}:1")

In [ ]:
import shutil, pathlib

# Grab one example image from a rare class
src_dir = pathlib.Path(TRAIN_DIR) / 'Paired_Doves'
example_files = list(src_dir.glob("*.png"))[:1]

if example_files:
    shutil.copy(example_files[0], '/kaggle/working/example_paired_doves.png')
    print(f"Copied: {example_files[0]}")
    print("Now go to the Kaggle sidebar → Output → find 'example_paired_doves.png' → click the download icon")
else:
    print("No files found — check the path")

In [ ]:
import pathlib
import numpy as np
from PIL import Image, ImageEnhance
import random

random.seed(42)

TRAIN_DIR_PATH = pathlib.Path(TRAIN_DIR)

# Classes that need help, and how many total images you want each to reach
AUGMENT_TARGETS = {
    'Paired_Doves': 300,
    'Wandering_Line': 300,
    'Chirp': 300,
    'Air_Compressor': 300,
    '1400Ripples': 300,
    'None_of_the_Above': 300,
}

def augment_image(img):
    """Apply one random augmentation — safe transforms for spectrograms
    (no vertical flip: frequency axis has real physical meaning, flipping it
    would create physically nonsensical glitches)"""
    choice = random.choice(['rotate', 'brightness', 'contrast', 'horizontal_flip', 'zoom'])

    if choice == 'rotate':
        angle = random.uniform(-10, 10)  # small angle only — big rotations distort spectrogram shape
        img = img.rotate(angle, fillcolor=(255, 255, 255))
    elif choice == 'brightness':
        factor = random.uniform(0.85, 1.15)
        img = ImageEnhance.Brightness(img).enhance(factor)
    elif choice == 'contrast':
        factor = random.uniform(0.85, 1.15)
        img = ImageEnhance.Contrast(img).enhance(factor)
    elif choice == 'horizontal_flip':
        img = img.transpose(Image.FLIP_LEFT_RIGHT)  # time-axis flip is physically reasonable
    elif choice == 'zoom':
        w, h = img.size
        crop_pct = random.uniform(0.9, 0.98)
        cw, ch = int(w * crop_pct), int(h * crop_pct)
        left = random.randint(0, w - cw)
        top = random.randint(0, h - ch)
        img = img.crop((left, top, left + cw, top + ch)).resize((w, h))

    return img

for class_name, target_count in AUGMENT_TARGETS.items():
    class_dir = TRAIN_DIR_PATH / class_name
    existing_files = list(class_dir.glob("*.png"))
    current_count = len(existing_files)

    if current_count >= target_count:
        print(f"{class_name}: already at {current_count}, skipping")
        continue

    needed = target_count - current_count
    print(f"{class_name}: {current_count} → {target_count} (generating {needed} augmented images)")

    for i in range(needed):
        source_file = random.choice(existing_files)
        img = Image.open(source_file).convert('RGB')
        augmented = augment_image(img)
        out_path = class_dir / f"aug_{i:04d}_{source_file.stem}.png"
        augmented.save(out_path)

print("\nDone. Re-run your EDA cell to confirm the new per-class train counts.")

In [2]:
# ── Cell 7: Train on the AUGMENTED data, same seeds as your baseline, for a fair comparison ──
import time, pathlib, numpy as np, pandas as pd, tensorflow as tf
from tensorflow.keras import layers
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

IMG_SIZE, BATCH, EPOCHS = (128, 128), 32, 15
SEEDS = [220301, 42, 7]   # same seeds as plus_lr_schedule baseline
autotune = tf.data.AUTOTUNE

data_dir = pathlib.Path(TRAIN_DIR)
class_names = sorted([p.name for p in data_dir.iterdir() if p.is_dir()])
num_classes = len(class_names)

# Recompute class weights — augmentation changed the counts for 6 classes
train_labels_list = []
for i, name in enumerate(class_names):
    train_labels_list.extend([i] * len(list((data_dir / name).glob("*.png"))))
train_labels_arr = np.array(train_labels_list)
raw_weights = compute_class_weight(class_weight="balanced", classes=np.unique(train_labels_arr), y=train_labels_arr)
capped_weights = np.clip(raw_weights, None, raw_weights.min() * 10.0)   # same cap as your baseline
class_weight_dict = dict(enumerate(capped_weights))

try:
    from classification_models.tfkeras import Classifiers
    ResNetBackbone, preprocess_input = Classifiers.get('resnet18')
    BACKBONE_NAME = "resnet18"
except Exception:
    from tensorflow.keras.applications import ResNet50 as ResNetBackbone
    from tensorflow.keras.applications.resnet50 import preprocess_input
    BACKBONE_NAME = "resnet50_fallback"

def build_model():
    if BACKBONE_NAME == "resnet18":
        base = ResNetBackbone(input_shape=IMG_SIZE + (3,), weights='imagenet', include_top=False)
    else:
        base = ResNetBackbone(include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,))
    base.trainable = False
    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = preprocess_input(inputs)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs)

all_results = []
for seed in SEEDS:
    print(f"\n{'='*60}\nwith_augmentation — seed {seed}\n{'='*60}")
    tf.random.set_seed(seed)

    train_ds_raw = tf.keras.utils.image_dataset_from_directory(TRAIN_DIR, seed=seed, image_size=IMG_SIZE, batch_size=BATCH)
    valid_ds_raw = tf.keras.utils.image_dataset_from_directory(VALID_DIR, seed=seed, image_size=IMG_SIZE, batch_size=BATCH)
    test_ds = tf.keras.utils.image_dataset_from_directory(TEST_DIR, seed=seed, image_size=IMG_SIZE, batch_size=BATCH)

    train_ds = train_ds_raw.map(lambda x, y: (x, tf.one_hot(y, num_classes)), num_parallel_calls=autotune).prefetch(autotune)
    valid_ds = valid_ds_raw.map(lambda x, y: (x, tf.one_hot(y, num_classes)), num_parallel_calls=autotune).prefetch(autotune)
    test_ds = test_ds.prefetch(autotune)

    model = build_model()
    model.compile(
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.3),
        optimizer=tf.keras.optimizers.RMSprop(learning_rate=1e-3),
        metrics=["accuracy"]
    )
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6)
    ]

    hist = model.fit(train_ds, validation_data=valid_ds, epochs=EPOCHS,
                      class_weight=class_weight_dict, callbacks=callbacks, verbose=0)

    y_true, y_pred = [], []
    for imgs, labels in test_ds:
        preds = model.predict(imgs, verbose=0)
        y_true.extend(labels.numpy())
        y_pred.extend(np.argmax(preds, axis=1))
    report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)

    result = {
        "variant": "with_augmentation", "seed": seed,
        "test_accuracy": round(report["accuracy"], 4),
        "macro_f1": round(report["macro avg"]["f1-score"], 4),
        "paired_doves_recall": round(report.get("Paired_Doves", {}).get("recall", float("nan")), 2),
        "no_glitch_recall": round(report.get("No_Glitch", {}).get("recall", float("nan")), 2),
    }
    all_results.append(result)
    print(result)

results_df = pd.DataFrame(all_results)
print("\n\n" + "="*80)
print("WITH AUGMENTATION — mean ± std across 3 seeds")
print("="*80)
print(results_df.agg(['mean', 'std']).to_string())
results_df.to_csv("/kaggle/working/augmentation_results.csv", index=False)

print("\n\n=== COMPARE AGAINST YOUR plus_lr_schedule BASELINE (no augmentation) ===")
print("plus_lr_schedule: test_acc=[0.8877, 0.8909, 0.8955], macro_f1=[0.8205, 0.8331, 0.8297], paired_doves_recall=[0.25, 0.42, 0.25]")


with_augmentation — seed 220301
Found 25485 files belonging to 22 classes.


I0000 00:00:1789227948.870129      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1789227948.873227      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 3176 files belonging to 22 classes.
Found 3207 files belonging to 22 classes.
44920640/44920640 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


I0000 00:00:1789227959.961336     150 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


{'variant': 'with_augmentation', 'seed': 220301, 'test_accuracy': 0.8949, 'macro_f1': 0.8196, 'paired_doves_recall': 0.25, 'no_glitch_recall': 0.47}

with_augmentation — seed 42
Found 25485 files belonging to 22 classes.
Found 3176 files belonging to 22 classes.
Found 3207 files belonging to 22 classes.
{'variant': 'with_augmentation', 'seed': 42, 'test_accuracy': 0.9002, 'macro_f1': 0.8328, 'paired_doves_recall': 0.25, 'no_glitch_recall': 0.6}

with_augmentation — seed 7
Found 25485 files belonging to 22 classes.
Found 3176 files belonging to 22 classes.
Found 3207 files belonging to 22 classes.
{'variant': 'with_augmentation', 'seed': 7, 'test_accuracy': 0.904, 'macro_f1': 0.8327, 'paired_doves_recall': 0.25, 'no_glitch_recall': 0.48}


WITH AUGMENTATION — mean ± std across 3 seeds


TypeError: Could not convert string 'with_augmentationwith_augmentationwith_augmentation' to numeric

In [3]:
numeric_cols = ['test_accuracy', 'macro_f1', 'paired_doves_recall', 'no_glitch_recall']
print(results_df[numeric_cols].agg(['mean', 'std']).to_string())

      test_accuracy  macro_f1  paired_doves_recall  no_glitch_recall
mean       0.899700  0.828367                 0.25          0.516667
std        0.004571  0.007592                 0.00          0.072342


In [6]:
# ── Cell 7 (fully fixed): Cleanup + train with literature-grounded augmentation ──
import time, pathlib, numpy as np, pandas as pd, tensorflow as tf
from tensorflow.keras import layers
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

# ── Step 0: Remove contaminated offline flip-augmented files from earlier (if any remain) ──
removed = 0
for class_dir in pathlib.Path(TRAIN_DIR).iterdir():
    if class_dir.is_dir():
        for f in class_dir.glob("aug_*.png"):
            f.unlink()
            removed += 1
print(f"Removed {removed} contaminated offline-augmented files\n")

IMG_SIZE, BATCH, EPOCHS = (128, 128), 32, 15
SEEDS = [220301, 42, 7]
autotune = tf.data.AUTOTUNE

data_dir = pathlib.Path(TRAIN_DIR)
class_names = sorted([p.name for p in data_dir.iterdir() if p.is_dir()])
num_classes = len(class_names)

# Total training images — needed to explicitly set steps_per_epoch
total_train_images = sum(len(list((data_dir / name).glob("*.png"))) for name in class_names)
steps_per_epoch = total_train_images // BATCH
print(f"Total training images: {total_train_images}, steps_per_epoch: {steps_per_epoch}\n")

# Class weights
train_labels_list = []
for i, name in enumerate(class_names):
    train_labels_list.extend([i] * len(list((data_dir / name).glob("*.png"))))
train_labels_arr = np.array(train_labels_list)
raw_weights = compute_class_weight(class_weight="balanced", classes=np.unique(train_labels_arr), y=train_labels_arr)
capped_weights = np.clip(raw_weights, None, raw_weights.min() * 10.0)
class_weight_dict = dict(enumerate(capped_weights))

try:
    from classification_models.tfkeras import Classifiers
    ResNetBackbone, preprocess_input = Classifiers.get('resnet18')
    BACKBONE_NAME = "resnet18"
except Exception:
    from tensorflow.keras.applications import ResNet50 as ResNetBackbone
    from tensorflow.keras.applications.resnet50 import preprocess_input
    BACKBONE_NAME = "resnet50_fallback"

# ── Augmentation — operates on SINGLE unbatched images (H, W, C) ──
def time_offset_augment(img, max_shift_px=8):
    shift = tf.random.uniform([], -max_shift_px, max_shift_px, dtype=tf.int32)
    return tf.roll(img, shift=shift, axis=1)

def spec_augment(img, time_mask_width=15, freq_mask_width=15):
    h, w = IMG_SIZE[0], IMG_SIZE[1]
    t0 = tf.random.uniform([], 0, w - time_mask_width, dtype=tf.int32)
    mask_t = tf.concat([tf.ones([h, t0, 3]), tf.zeros([h, time_mask_width, 3]),
                         tf.ones([h, w - t0 - time_mask_width, 3])], axis=1)
    f0 = tf.random.uniform([], 0, h - freq_mask_width, dtype=tf.int32)
    mask_f = tf.concat([tf.ones([f0, w, 3]), tf.zeros([freq_mask_width, w, 3]),
                         tf.ones([h - f0 - freq_mask_width, w, 3])], axis=0)
    return img * mask_t * mask_f

def build_model():
    if BACKBONE_NAME == "resnet18":
        base = ResNetBackbone(input_shape=IMG_SIZE + (3,), weights='imagenet', include_top=False)
    else:
        base = ResNetBackbone(include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,))
    base.trainable = False
    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = preprocess_input(inputs)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs)

all_results = []
for seed in SEEDS:
    print(f"\n{'='*60}\nliterature_augmentation — seed {seed}\n{'='*60}")
    tf.random.set_seed(seed)

    train_ds_raw = tf.keras.utils.image_dataset_from_directory(TRAIN_DIR, seed=seed, image_size=IMG_SIZE, batch_size=BATCH)
    valid_ds_raw = tf.keras.utils.image_dataset_from_directory(VALID_DIR, seed=seed, image_size=IMG_SIZE, batch_size=BATCH)
    test_ds = tf.keras.utils.image_dataset_from_directory(TEST_DIR, seed=seed, image_size=IMG_SIZE, batch_size=BATCH)

    # unbatch → augment per-image → repeat (so it never runs dry) → re-batch
    train_ds = train_ds_raw.unbatch()
    train_ds = train_ds.map(lambda x, y: (x, tf.one_hot(y, num_classes)), num_parallel_calls=autotune)
    train_ds = train_ds.map(lambda x, y: (time_offset_augment(spec_augment(x)), y), num_parallel_calls=autotune)
    train_ds = train_ds.repeat().batch(BATCH).prefetch(autotune)

    valid_ds = valid_ds_raw.map(lambda x, y: (x, tf.one_hot(y, num_classes)), num_parallel_calls=autotune).prefetch(autotune)
    test_ds = test_ds.prefetch(autotune)

    model = build_model()
    model.compile(
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.3),
        optimizer=tf.keras.optimizers.RMSprop(learning_rate=1e-3),
        metrics=["accuracy"]
    )
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6)
    ]

    hist = model.fit(train_ds, validation_data=valid_ds, epochs=EPOCHS,
                      steps_per_epoch=steps_per_epoch,
                      class_weight=class_weight_dict, callbacks=callbacks, verbose=0)

    y_true, y_pred = [], []
    for imgs, labels in test_ds:
        preds = model.predict(imgs, verbose=0)
        y_true.extend(labels.numpy())
        y_pred.extend(np.argmax(preds, axis=1))
    report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)

    result = {
        "variant": "literature_augmentation", "seed": seed,
        "test_accuracy": round(report["accuracy"], 4),
        "macro_f1": round(report["macro avg"]["f1-score"], 4),
        "paired_doves_recall": round(report.get("Paired_Doves", {}).get("recall", float("nan")), 2),
        "no_glitch_recall": round(report.get("No_Glitch", {}).get("recall", float("nan")), 2),
    }
    all_results.append(result)
    print(result)

results_df = pd.DataFrame(all_results)
print("\n\n" + "="*80)
print("LITERATURE-GROUNDED AUGMENTATION — mean ± std across 3 seeds")
print("="*80)
numeric_cols = ['test_accuracy', 'macro_f1', 'paired_doves_recall', 'no_glitch_recall']
print(results_df[numeric_cols].agg(['mean', 'std']).to_string())
results_df.to_csv("/kaggle/working/literature_augmentation_results.csv", index=False)

print("\n\n=== COMPARE AGAINST plus_lr_schedule BASELINE (no augmentation) ===")
print("macro_f1: mean=0.828, std=0.007")

Removed 0 contaminated offline-augmented files

Total training images: 25485, steps_per_epoch: 796


literature_augmentation — seed 220301
Found 25485 files belonging to 22 classes.
Found 3176 files belonging to 22 classes.
Found 3207 files belonging to 22 classes.
{'variant': 'literature_augmentation', 'seed': 220301, 'test_accuracy': 0.6052, 'macro_f1': 0.3958, 'paired_doves_recall': 0.0, 'no_glitch_recall': 0.0}

literature_augmentation — seed 42
Found 25485 files belonging to 22 classes.
Found 3176 files belonging to 22 classes.
Found 3207 files belonging to 22 classes.
{'variant': 'literature_augmentation', 'seed': 42, 'test_accuracy': 0.5553, 'macro_f1': 0.398, 'paired_doves_recall': 0.0, 'no_glitch_recall': 0.0}

literature_augmentation — seed 7
Found 25485 files belonging to 22 classes.
Found 3176 files belonging to 22 classes.
Found 3207 files belonging to 22 classes.
{'variant': 'literature_augmentation', 'seed': 7, 'test_accuracy': 0.6009, 'macro_f1': 0.4169, 'paired_doves_r

In [7]:
# ── Diagnostic: same pipeline, augmentation DISABLED — isolates whether the bug is in
#    the unbatch/repeat restructuring itself, or in the augmentation content ──
print(f"\n{'='*60}\npipeline_check_no_augmentation — seed 220301\n{'='*60}")
tf.random.set_seed(220301)

train_ds_raw = tf.keras.utils.image_dataset_from_directory(TRAIN_DIR, seed=220301, image_size=IMG_SIZE, batch_size=BATCH)
valid_ds_raw = tf.keras.utils.image_dataset_from_directory(VALID_DIR, seed=220301, image_size=IMG_SIZE, batch_size=BATCH)
test_ds = tf.keras.utils.image_dataset_from_directory(TEST_DIR, seed=220301, image_size=IMG_SIZE, batch_size=BATCH)

# Same unbatch/repeat/re-batch structure as before — but NO augmentation applied
train_ds = train_ds_raw.unbatch()
train_ds = train_ds.map(lambda x, y: (x, tf.one_hot(y, num_classes)), num_parallel_calls=autotune)
# augmentation line intentionally removed for this diagnostic
train_ds = train_ds.repeat().batch(BATCH).prefetch(autotune)

valid_ds = valid_ds_raw.map(lambda x, y: (x, tf.one_hot(y, num_classes)), num_parallel_calls=autotune).prefetch(autotune)
test_ds = test_ds.prefetch(autotune)

model = build_model()
model.compile(
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.3),
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=1e-3),
    metrics=["accuracy"]
)
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6)
]

hist = model.fit(train_ds, validation_data=valid_ds, epochs=EPOCHS,
                  steps_per_epoch=steps_per_epoch,
                  class_weight=class_weight_dict, callbacks=callbacks, verbose=0)

y_true, y_pred = [], []
for imgs, labels in test_ds:
    preds = model.predict(imgs, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))
report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)

print({
    "variant": "pipeline_check_no_augmentation",
    "test_accuracy": round(report["accuracy"], 4),
    "macro_f1": round(report["macro avg"]["f1-score"], 4),
    "paired_doves_recall": round(report.get("Paired_Doves", {}).get("recall", float("nan")), 2),
    "no_glitch_recall": round(report.get("No_Glitch", {}).get("recall", float("nan")), 2),
})

print("\nCOMPARE: baseline macro_f1 = 0.828 (from plus_lr_schedule, original non-unbatch pipeline)")


pipeline_check_no_augmentation — seed 220301
Found 25485 files belonging to 22 classes.
Found 3176 files belonging to 22 classes.
Found 3207 files belonging to 22 classes.
{'variant': 'pipeline_check_no_augmentation', 'test_accuracy': 0.9021, 'macro_f1': 0.8368, 'paired_doves_recall': 0.5, 'no_glitch_recall': 0.58}

COMPARE: baseline macro_f1 = 0.828 (from plus_lr_schedule, original non-unbatch pipeline)


In [8]:
# ── Cell: Literature-grounded augmentation, TONED DOWN (fixes the too-aggressive masking) ──
import time, pathlib, numpy as np, pandas as pd, tensorflow as tf
from tensorflow.keras import layers
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

IMG_SIZE, BATCH, EPOCHS = (128, 128), 32, 15
SEEDS = [220301, 42, 7]
autotune = tf.data.AUTOTUNE

data_dir = pathlib.Path(TRAIN_DIR)
class_names = sorted([p.name for p in data_dir.iterdir() if p.is_dir()])
num_classes = len(class_names)

total_train_images = sum(len(list((data_dir / name).glob("*.png"))) for name in class_names)
steps_per_epoch = total_train_images // BATCH
print(f"Total training images: {total_train_images}, steps_per_epoch: {steps_per_epoch}\n")

train_labels_list = []
for i, name in enumerate(class_names):
    train_labels_list.extend([i] * len(list((data_dir / name).glob("*.png"))))
train_labels_arr = np.array(train_labels_list)
raw_weights = compute_class_weight(class_weight="balanced", classes=np.unique(train_labels_arr), y=train_labels_arr)
capped_weights = np.clip(raw_weights, None, raw_weights.min() * 10.0)
class_weight_dict = dict(enumerate(capped_weights))

try:
    from classification_models.tfkeras import Classifiers
    ResNetBackbone, preprocess_input = Classifiers.get('resnet18')
    BACKBONE_NAME = "resnet18"
except Exception:
    from tensorflow.keras.applications import ResNet50 as ResNetBackbone
    from tensorflow.keras.applications.resnet50 import preprocess_input
    BACKBONE_NAME = "resnet50_fallback"

# ── TONED DOWN augmentation — smaller shift and mask sizes ──
def time_offset_augment(img, max_shift_px=4):        # was 8
    shift = tf.random.uniform([], -max_shift_px, max_shift_px, dtype=tf.int32)
    return tf.roll(img, shift=shift, axis=1)

def spec_augment(img, time_mask_width=6, freq_mask_width=6):   # was 15
    h, w = IMG_SIZE[0], IMG_SIZE[1]
    t0 = tf.random.uniform([], 0, w - time_mask_width, dtype=tf.int32)
    mask_t = tf.concat([tf.ones([h, t0, 3]), tf.zeros([h, time_mask_width, 3]),
                         tf.ones([h, w - t0 - time_mask_width, 3])], axis=1)
    f0 = tf.random.uniform([], 0, h - freq_mask_width, dtype=tf.int32)
    mask_f = tf.concat([tf.ones([f0, w, 3]), tf.zeros([freq_mask_width, w, 3]),
                         tf.ones([h - f0 - freq_mask_width, w, 3])], axis=0)
    return img * mask_t * mask_f

def build_model():
    if BACKBONE_NAME == "resnet18":
        base = ResNetBackbone(input_shape=IMG_SIZE + (3,), weights='imagenet', include_top=False)
    else:
        base = ResNetBackbone(include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,))
    base.trainable = False
    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = preprocess_input(inputs)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs)

all_results = []
for seed in SEEDS:
    print(f"\n{'='*60}\nliterature_augmentation_v2 — seed {seed}\n{'='*60}")
    tf.random.set_seed(seed)

    train_ds_raw = tf.keras.utils.image_dataset_from_directory(TRAIN_DIR, seed=seed, image_size=IMG_SIZE, batch_size=BATCH)
    valid_ds_raw = tf.keras.utils.image_dataset_from_directory(VALID_DIR, seed=seed, image_size=IMG_SIZE, batch_size=BATCH)
    test_ds = tf.keras.utils.image_dataset_from_directory(TEST_DIR, seed=seed, image_size=IMG_SIZE, batch_size=BATCH)

    train_ds = train_ds_raw.unbatch()
    train_ds = train_ds.map(lambda x, y: (x, tf.one_hot(y, num_classes)), num_parallel_calls=autotune)
    train_ds = train_ds.map(lambda x, y: (time_offset_augment(spec_augment(x)), y), num_parallel_calls=autotune)
    train_ds = train_ds.repeat().batch(BATCH).prefetch(autotune)

    valid_ds = valid_ds_raw.map(lambda x, y: (x, tf.one_hot(y, num_classes)), num_parallel_calls=autotune).prefetch(autotune)
    test_ds = test_ds.prefetch(autotune)

    model = build_model()
    model.compile(
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.3),
        optimizer=tf.keras.optimizers.RMSprop(learning_rate=1e-3),
        metrics=["accuracy"]
    )
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6)
    ]

    hist = model.fit(train_ds, validation_data=valid_ds, epochs=EPOCHS,
                      steps_per_epoch=steps_per_epoch,
                      class_weight=class_weight_dict, callbacks=callbacks, verbose=0)

    y_true, y_pred = [], []
    for imgs, labels in test_ds:
        preds = model.predict(imgs, verbose=0)
        y_true.extend(labels.numpy())
        y_pred.extend(np.argmax(preds, axis=1))
    report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)

    result = {
        "variant": "literature_augmentation_v2", "seed": seed,
        "test_accuracy": round(report["accuracy"], 4),
        "macro_f1": round(report["macro avg"]["f1-score"], 4),
        "paired_doves_recall": round(report.get("Paired_Doves", {}).get("recall", float("nan")), 2),
        "no_glitch_recall": round(report.get("No_Glitch", {}).get("recall", float("nan")), 2),
    }
    all_results.append(result)
    print(result)

results_df = pd.DataFrame(all_results)
print("\n\n" + "="*80)
print("LITERATURE AUGMENTATION v2 (toned down) — mean ± std across 3 seeds")
print("="*80)
numeric_cols = ['test_accuracy', 'macro_f1', 'paired_doves_recall', 'no_glitch_recall']
print(results_df[numeric_cols].agg(['mean', 'std']).to_string())
results_df.to_csv("/kaggle/working/literature_augmentation_v2_results.csv", index=False)

print("\n\n=== COMPARE ===")
print("no_augmentation baseline (plus_lr_schedule): macro_f1 mean=0.828, std=0.007")
print("no_augmentation (same pipeline, this run's structure): macro_f1=0.8368")
print("v1 augmentation (too aggressive, mask=15,shift=8): macro_f1 mean=0.4036")

Total training images: 25485, steps_per_epoch: 796


literature_augmentation_v2 — seed 220301
Found 25485 files belonging to 22 classes.
Found 3176 files belonging to 22 classes.
Found 3207 files belonging to 22 classes.
{'variant': 'literature_augmentation_v2', 'seed': 220301, 'test_accuracy': 0.7041, 'macro_f1': 0.5434, 'paired_doves_recall': 0.0, 'no_glitch_recall': 0.0}

literature_augmentation_v2 — seed 42
Found 25485 files belonging to 22 classes.
Found 3176 files belonging to 22 classes.
Found 3207 files belonging to 22 classes.
{'variant': 'literature_augmentation_v2', 'seed': 42, 'test_accuracy': 0.6894, 'macro_f1': 0.5178, 'paired_doves_recall': 0.0, 'no_glitch_recall': 0.03}

literature_augmentation_v2 — seed 7
Found 25485 files belonging to 22 classes.
Found 3176 files belonging to 22 classes.
Found 3207 files belonging to 22 classes.
{'variant': 'literature_augmentation_v2', 'seed': 7, 'test_accuracy': 0.6791, 'macro_f1': 0.4854, 'paired_doves_recall': 0.0, 'no_glitch_reca

In [9]:
# ── Cell: TIME-OFFSET ONLY (no masking) — isolates whether spec_augment is the real culprit ──
import time, pathlib, numpy as np, pandas as pd, tensorflow as tf
from tensorflow.keras import layers
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

IMG_SIZE, BATCH, EPOCHS = (128, 128), 32, 15
SEEDS = [220301, 42, 7]
autotune = tf.data.AUTOTUNE

data_dir = pathlib.Path(TRAIN_DIR)
class_names = sorted([p.name for p in data_dir.iterdir() if p.is_dir()])
num_classes = len(class_names)

total_train_images = sum(len(list((data_dir / name).glob("*.png"))) for name in class_names)
steps_per_epoch = total_train_images // BATCH
print(f"Total training images: {total_train_images}, steps_per_epoch: {steps_per_epoch}\n")

train_labels_list = []
for i, name in enumerate(class_names):
    train_labels_list.extend([i] * len(list((data_dir / name).glob("*.png"))))
train_labels_arr = np.array(train_labels_list)
raw_weights = compute_class_weight(class_weight="balanced", classes=np.unique(train_labels_arr), y=train_labels_arr)
capped_weights = np.clip(raw_weights, None, raw_weights.min() * 10.0)
class_weight_dict = dict(enumerate(capped_weights))

try:
    from classification_models.tfkeras import Classifiers
    ResNetBackbone, preprocess_input = Classifiers.get('resnet18')
    BACKBONE_NAME = "resnet18"
except Exception:
    from tensorflow.keras.applications import ResNet50 as ResNetBackbone
    from tensorflow.keras.applications.resnet50 import preprocess_input
    BACKBONE_NAME = "resnet50_fallback"

def time_offset_augment(img, max_shift_px=4):
    shift = tf.random.uniform([], -max_shift_px, max_shift_px, dtype=tf.int32)
    return tf.roll(img, shift=shift, axis=1)

def build_model():
    if BACKBONE_NAME == "resnet18":
        base = ResNetBackbone(input_shape=IMG_SIZE + (3,), weights='imagenet', include_top=False)
    else:
        base = ResNetBackbone(include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,))
    base.trainable = False
    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = preprocess_input(inputs)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs)

all_results = []
for seed in SEEDS:
    print(f"\n{'='*60}\ntime_offset_only — seed {seed}\n{'='*60}")
    tf.random.set_seed(seed)

    train_ds_raw = tf.keras.utils.image_dataset_from_directory(TRAIN_DIR, seed=seed, image_size=IMG_SIZE, batch_size=BATCH)
    valid_ds_raw = tf.keras.utils.image_dataset_from_directory(VALID_DIR, seed=seed, image_size=IMG_SIZE, batch_size=BATCH)
    test_ds = tf.keras.utils.image_dataset_from_directory(TEST_DIR, seed=seed, image_size=IMG_SIZE, batch_size=BATCH)

    train_ds = train_ds_raw.unbatch()
    train_ds = train_ds.map(lambda x, y: (x, tf.one_hot(y, num_classes)), num_parallel_calls=autotune)
    train_ds = train_ds.map(lambda x, y: (time_offset_augment(x), y), num_parallel_calls=autotune)  # ONLY time-offset, no masking
    train_ds = train_ds.repeat().batch(BATCH).prefetch(autotune)

    valid_ds = valid_ds_raw.map(lambda x, y: (x, tf.one_hot(y, num_classes)), num_parallel_calls=autotune).prefetch(autotune)
    test_ds = test_ds.prefetch(autotune)

    model = build_model()
    model.compile(
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.3),
        optimizer=tf.keras.optimizers.RMSprop(learning_rate=1e-3),
        metrics=["accuracy"]
    )
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6)
    ]

    hist = model.fit(train_ds, validation_data=valid_ds, epochs=EPOCHS,
                      steps_per_epoch=steps_per_epoch,
                      class_weight=class_weight_dict, callbacks=callbacks, verbose=0)

    y_true, y_pred = [], []
    for imgs, labels in test_ds:
        preds = model.predict(imgs, verbose=0)
        y_true.extend(labels.numpy())
        y_pred.extend(np.argmax(preds, axis=1))
    report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)

    result = {
        "variant": "time_offset_only", "seed": seed,
        "test_accuracy": round(report["accuracy"], 4),
        "macro_f1": round(report["macro avg"]["f1-score"], 4),
        "paired_doves_recall": round(report.get("Paired_Doves", {}).get("recall", float("nan")), 2),
        "no_glitch_recall": round(report.get("No_Glitch", {}).get("recall", float("nan")), 2),
    }
    all_results.append(result)
    print(result)

results_df = pd.DataFrame(all_results)
print("\n\n" + "="*80)
print("TIME-OFFSET ONLY — mean ± std across 3 seeds")
print("="*80)
numeric_cols = ['test_accuracy', 'macro_f1', 'paired_doves_recall', 'no_glitch_recall']
print(results_df[numeric_cols].agg(['mean', 'std']).to_string())
results_df.to_csv("/kaggle/working/time_offset_only_results.csv", index=False)

print("\n\n=== COMPARE ===")
print("no_augmentation (pipeline check): macro_f1=0.8368")
print("no_augmentation baseline (plus_lr_schedule): macro_f1 mean=0.828, std=0.007")
print("v1 augmentation (mask=15,shift=8): macro_f1 mean=0.4036")
print("v2 augmentation (mask=6,shift=4): macro_f1 mean=0.5155")

Total training images: 25485, steps_per_epoch: 796


time_offset_only — seed 220301
Found 25485 files belonging to 22 classes.
Found 3176 files belonging to 22 classes.
Found 3207 files belonging to 22 classes.
{'variant': 'time_offset_only', 'seed': 220301, 'test_accuracy': 0.8899, 'macro_f1': 0.8191, 'paired_doves_recall': 0.25, 'no_glitch_recall': 0.62}

time_offset_only — seed 42
Found 25485 files belonging to 22 classes.
Found 3176 files belonging to 22 classes.
Found 3207 files belonging to 22 classes.
{'variant': 'time_offset_only', 'seed': 42, 'test_accuracy': 0.9036, 'macro_f1': 0.8375, 'paired_doves_recall': 0.33, 'no_glitch_recall': 0.53}

time_offset_only — seed 7
Found 25485 files belonging to 22 classes.
Found 3176 files belonging to 22 classes.
Found 3207 files belonging to 22 classes.
{'variant': 'time_offset_only', 'seed': 7, 'test_accuracy': 0.9018, 'macro_f1': 0.8246, 'paired_doves_recall': 0.25, 'no_glitch_recall': 0.57}


TIME-OFFSET ONLY — mean ± std across 3 seed